# Quickstart: OpenAI Responses tool calling with Spark

Use `OpenAIPrompt` to ask for function calls, resolve them with ordinary Spark operations, and use `OpenAIResponses` for an explicit continuation. SynapseML transports tool declarations, calls, and outputs; it never executes a function or runs an automatic tool loop.


## Safety, trust, and execution semantics

Tool definitions are **trusted pipeline configuration**. Do not populate `toolsCol` from untrusted row data: hosted or MCP tools can cause provider-side network access and cost. Model-produced `arguments` are untrusted strings, so validate them with `from_json` and an allowlist before joining or calling anything external. Truncate tool outputs before returning them.

Spark plus HTTP is **at-least-once**: task retry, executor loss, speculation, ambiguous HTTP retry, or recomputing an unmaterialized lineage can repeat paid requests. Disable speculation around paid stages, materialize each turn before branching, use durable `checkpoint()` or write/read in production, deduplicate costs by `response.id`, and make external side effects idempotent by `call_id`.


## Configure the service

This sample uses a stored continuation (`store=True` plus `previous_response_id`). Azure dated endpoints support core function calling; newer fields may require an `/openai/v1` URL. `serviceTier` is not supported by Azure OpenAI. Streaming, background response lifecycle, automatic tool execution, and moderation configuration are intentionally outside this synchronous transformer.


In [ ]:
from synapse.ml.core.platform import find_secret
from synapse.ml.services.openai import OpenAIPrompt, OpenAIResponses
from pyspark.sql import functions as F

service_name = "synapseml-openai-3"
deployment_name = "gpt-5.1"
api_version = "2025-04-01-preview"
key = find_secret(secret_name="openai-api-key-3", keyvault="mmlspark-build-keys")

# Configure spark.speculation=false when creating the Spark session or cluster.
# Managed runtimes such as Databricks may not allow changing it at runtime.

## Define trusted tools and input rows

Function tools use the flat Responses shape. Nested Chat Completions-style function definitions are accepted and normalized, but no Chat Completions tool adapter is involved.


In [ ]:
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a city.",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
        "additionalProperties": False,
    },
    "strict": True,
}

cities = spark.createDataFrame([(1, "Seattle"), (2, "Boston")], ["row_id", "city"])

## Turn 1: ask with `OpenAIPrompt`

`toolCallsCol` is opt-in structured data. `responseStructCol` retains the parsed Responses struct, and `responseIdCol` is available because this stored flow sets `store=True`. Text remains the default Prompt output; it is null on rows where the model asks for a tool.


In [ ]:
turn1 = (
    OpenAIPrompt()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setApiType("responses")
    .setPromptTemplate("What is the weather in {city}? Use the tool.")
    .setTools([weather_tool])
    .setToolChoice("auto")
    .setParallelToolCalls(True)
    .setStore(True)
    .setResponseIdCol("response_id")
    .setToolCallsCol("tool_calls")
    .setResponseStructCol("response_struct")
    .setOutputCol("answer")
)

asked = turn1.transform(cities).persist()
asked.count()  # Materialize this paid turn before branching.
asked.select("row_id", "city", "answer", "tool_calls").show(truncate=False)

## Validate arguments, then resolve externally

Never use `eval`, dynamic SQL identifiers, or arbitrary dispatch on model output. Parse with a fixed schema, allowlist the declared function name, reject incomplete arguments, and join to a controlled table or service result. This sample uses a Spark table so no code executes inside the OpenAI transformer.


In [ ]:
argument_schema = "city STRING"
allowed_function_names = ["get_weather"]

calls = (
    asked.select(
        "row_id",
        "response_id",
        F.explode("tool_calls").alias("call"),
    )
    .where(F.col("call.name").isin(allowed_function_names))
    .withColumn("arguments", F.from_json("call.arguments", argument_schema))
    .where(
        F.col("arguments").isNotNull()
        & F.col("arguments.city").isNotNull()
        & (F.length("arguments.city") <= 64)
    )
    .withColumn("requested_city", F.col("arguments.city"))
)

weather_table = spark.createDataFrame(
    [("Seattle", 20.0), ("Boston", 24.0)], ["city", "temp_c"]
)
resolved = calls.join(weather_table, calls.requested_city == weather_table.city, "left")

## Build typed function outputs

`functionCallOutputsCol` expects `ARRAY<STRUCT<call_id, output, status>>`. `output` is an opaque string, so serialize a bounded result and truncate it before returning it to the model. External operations with side effects must be idempotent by `call_id`.


In [ ]:
tool_results = resolved.groupBy("row_id", "response_id").agg(
    F.collect_list(
        F.struct(
            F.col("call.call_id").alias("call_id"),
            F.substring(
                F.to_json(
                    F.struct(
                        F.col("requested_city").alias("city"),
                        F.col("temp_c"),
                    )
                ),
                1,
                4000,
            ).alias("output"),
            F.lit("completed").alias("status"),
        )
    ).alias("function_outputs")
)

## Turn 2: explicit continuation

Tools are not carried across turns, so resend the same trusted definitions. Rows without function outputs are skipped and issue no second request. Materialize the second paid turn before reusing it. For a stateless flow, keep the original user messages, build `inputItemsCol` with `replayItemsColumn`, use `store=False`, and send messages → replay items → function outputs in that order.


In [ ]:
turn2 = (
    OpenAIResponses()
    .setSubscriptionKey(key)
    .setDeploymentName(deployment_name)
    .setCustomServiceName(service_name)
    .setApiVersion(api_version)
    .setFunctionCallOutputsCol("function_outputs")
    .setPreviousResponseIdCol("response_id")
    .setTools([weather_tool])
    .setOutputCol("response2")
)

answered = turn2.transform(tool_results).persist()
answered.count()
answered.selectExpr(
    "row_id",
    "element_at(filter(response2.output, x -> x.type = 'message'), -1).content[0].text AS answer",
).show(truncate=False)